In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
from utils.model_loader import get_model_fits
import numpy as np
import pandas as pd
import re
from sklearn.metrics import mean_squared_error
import seaborn as sns
import matplotlib.pyplot as plt


In [ ]:
data_dir = "datasets/friedman"
results_dir = "results/regression/single_layer/tanh/friedman"
results_dir_corr = "results/regression/single_layer/tanh/friedman_correlated"

model_names = ["Gaussian", "RHS", "DHS", "DST"]

fits, fits_corr = {}, {}
for fname in sorted(f for f in os.listdir(data_dir) if f.endswith(".npz") and "_N50_" not in f):
    config = fname.replace(".npz", "")
    fit = get_model_fits(config=config, results_dir=results_dir,
                         models=model_names, include_prior=False)
    if fit:
        fits[config] = fit

data_dir_corr = "datasets/friedman_correlated"
for fname in sorted(f for f in os.listdir(data_dir_corr) if f.endswith(".npz")):
    config = fname.replace(".npz", "")
    fit = get_model_fits(config=config, results_dir=results_dir_corr,
                         models=model_names, include_prior=False)
    if fit:
        fits_corr[config] = fit


In [ ]:
from utils.generate_data import generate_Friedman_data, generate_correlated_Friedman_data
from utils.sparsity import forward_pass_tanh

_KEY = re.compile(r"Friedman_N(\d+)_p\d+_sigma([\d.]+)_seed(\d+)")

def parse_key(key):
    m = _KEY.match(key)
    return int(m.group(1)), float(m.group(2)), int(m.group(3))

def crps_from_samples(y_test, y_pred):
    """Mean CRPS; y_pred shape (S, N_test)."""
    scores = []
    for n in range(len(y_test)):
        s = y_pred[:, n]
        scores.append(np.mean(np.abs(s - y_test[n])) - 0.5 * np.mean(np.abs(s[:, None] - s[None, :])))
    return float(np.mean(scores))

def evaluate(fits_dict, data_func, models, N_test=2000):
    """Compute RMSE and CRPS using a fresh large test set via forward pass."""
    rows = []
    for config, model_fits in fits_dict.items():
        N, sigma, seed = parse_key(config)
        # Regenerate with original seed to get same standardisation, then use large X_test
        _, _, y_train, _ = data_func(N=N, D=10, sigma=sigma, seed=seed)
        y_mean_train = y_train.mean(); y_std_train = y_train.std()
        # Fresh large test set (unstandardised X, raw y)
        _, X_test, _, y_test_raw = data_func(N=N_test, D=10, sigma=sigma, seed=seed + 999)
        # Standardise y the same way training did
        y_test = (y_test_raw - y_mean_train) / y_std_train

        for model, entry in model_fits.items():
            if "posterior" not in entry:
                continue
            post = entry["posterior"]
            W1 = post.stan_variable("W_1")        # (S, P, H)
            b1 = post.stan_variable("hidden_bias") # (S, 1, H)
            W2 = post.stan_variable("W_L")         # (S, H, 1)
            b2 = post.stan_variable("output_bias") # (S, 1)

            S = W1.shape[0]
            y_samps = np.zeros((S, len(y_test)))
            for s in range(S):
                y_hat = forward_pass_tanh(
                    X_test,
                    W1[s],
                    b1[s].reshape(-1),
                    W2[s],
                    b2[s].reshape(-1),
                )
                y_samps[s] = y_hat.squeeze()

            y_pred_mean = y_samps.mean(axis=0)
            rmse = float(np.sqrt(mean_squared_error(y_test, y_pred_mean)))
            crps = crps_from_samples(y_test, y_samps)
            rows.append(dict(config=config, N=N, sigma=sigma, seed=seed,
                             model=model, rmse=rmse, crps=crps))
    return pd.DataFrame(rows)


In [ ]:
df_orig = evaluate(fits,      generate_Friedman_data,           model_names)
df_corr = evaluate(fits_corr, generate_correlated_Friedman_data, model_names)
df_orig['setting'] = 'Independent'
df_corr['setting'] = 'Correlated'
df = pd.concat([df_orig, df_corr], ignore_index=True)


## Table 1 — Posterior mean RMSE

In [ ]:
table = (
    df.groupby(['setting', 'N', 'model'])['rmse']
      .agg(mean='mean', std='std')
      .reset_index()
)
table['cell'] = table.apply(lambda r: f"{r['mean']:.3f} ({r['std']:.3f})", axis=1)
pivot = table.pivot_table(index='model', columns=['setting', 'N'], values='cell', aggfunc='first')
print(pivot.to_string())


## Figure 5 — CRPS boxplots


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=False)
for ax, setting in zip(axes, ['Independent', 'Correlated']):
    sub = df[(df['setting'] == setting) & (df['N'].isin([100, 200, 500]))]
    sns.boxplot(data=sub, x='model', y='crps', hue='N',
                order=model_names, hue_order=[100, 200, 500],
                ax=ax, palette='Blues')
    ax.set_title(f'{setting} Friedman')
    ax.set_xlabel('')
    ax.set_ylabel('CRPS')
    ax.legend(title='N', fontsize=9)
plt.tight_layout()
plt.savefig('figures_for_use_in_paper/fig5_friedman_crps.pdf', bbox_inches='tight')
plt.show()


## Figure 6 — Effective parameters (m_eff)
See `kappa_matrix_clean.ipynb`.
